Exercise 1: Blockchain Structure and Tamper Detection

In [ ]:
"""Simple blockchain with SHA-256 block hashes and a Merkle Root per block."""

import hashlib
import json
import time

class Block:

    def __init__(self, index, transactions, previous_hash, timestamp=None):
        self.index = index
        self.timestamp = timestamp if timestamp is not None else time.time()
        self.transactions = transactions          # list of dicts
        self.previous_hash = previous_hash        # link to the parent block

        # Merkle Root is computed once and STORED, so tampering with
        # transactions later is detectable by recomputing and comparing.
        self.merkle_root = self.compute_merkle_root()
        self.hash = self.compute_hash()

    # ---------- hashing helpers ----------
    @staticmethod
    def _sha256(data: str) -> str:
        return hashlib.sha256(data.encode("utf-8")).hexdigest()

    @staticmethod
    def _serialise_tx(tx) -> str:
        # sort_keys makes the JSON deterministic -> same tx always gives same hash
        return json.dumps(tx, sort_keys=True)

    def compute_merkle_root(self) -> str:
        """Build the Merkle tree bottom-up and return the root hash."""
        if not self.transactions:
            return self._sha256("")

        # Leaf level: hash of each transaction
        level = [self._sha256(self._serialise_tx(tx)) for tx in self.transactions]

        # Combine pairs until one hash remains
        while len(level) > 1:
            if len(level) % 2 == 1:
                level.append(level[-1])   # odd count: duplicate last (Bitcoin rule)
            level = [self._sha256(level[i] + level[i + 1])
                     for i in range(0, len(level), 2)]
        return level[0]

    def compute_hash(self) -> str:
        """Hash of the block header. Uses merkle_root instead of raw
        transactions, so any tx change must flow through the Merkle Root."""
        header = {
            "index": self.index,
            "timestamp": self.timestamp,
            "merkle_root": self.merkle_root,
            "previous_hash": self.previous_hash,
        }
        return self._sha256(json.dumps(header, sort_keys=True))

    def __str__(self):
        tx_lines = "\n".join(
            f"      - {tx['sender']} -> {tx['receiver']}: {tx['amount']}"
            for tx in self.transactions
        )
        return (
            f"Block #{self.index} (block {self.index + 1} in chain)\n"
            f"  Timestamp     : {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(self.timestamp))}\n"
            f"  Transactions  :\n{tx_lines}\n"
            f"  Merkle Root   : {self.merkle_root}\n"
            f"  Previous Hash : {self.previous_hash}\n"
            f"  Hash          : {self.hash}\n"
        )


In [ ]:

class Blockchain:
    """Holds the list of blocks and validates the whole chain."""

    def __init__(self):
        self._chain = []                 # "private" by convention (like C# private field)
        self._create_genesis_block()     # genesis is created automatically

    def _create_genesis_block(self):
        genesis_tx = [{"sender": "Network", "receiver": "Genesis", "amount": 0}]
        # previous_hash of genesis is 64 zeros since it has no parent
        self._chain.append(Block(0, genesis_tx, "0" * 64))

    @property
    def chain(self):
        return self._chain

    @property
    def last_block(self):
        return self._chain[-1]

    def add_block(self, transactions):
        new_block = Block(len(self._chain), transactions, self.last_block.hash)
        self._chain.append(new_block)
        return new_block

    def validate(self):
        """Check every block and return a report of the FIRST failure.
        Three checks per block:
          1. Merkle Root still matches the transactions (data integrity)
          2. Stored hash still matches the header (header integrity)
          3. previous_hash matches the parent's hash (chain linkage)
        """
        for i, block in enumerate(self._chain):
            recomputed_root = block.compute_merkle_root()
            if recomputed_root != block.merkle_root:
                return self._failure(block, "Merkle Root mismatch - transaction data was modified",
                                     block.merkle_root, recomputed_root)

            recomputed_hash = block.compute_hash()
            if recomputed_hash != block.hash:
                return self._failure(block, "Block hash mismatch - block header was modified",
                                     block.hash, recomputed_hash)

            if i > 0 and block.previous_hash != self._chain[i - 1].hash:
                return self._failure(block, "Broken link - previous_hash does not match parent block's hash",
                                     block.previous_hash, self._chain[i - 1].hash)

        return {"valid": True, "blocks_checked": len(self._chain)}

    def _failure(self, block, reason, stored, expected):
        # Every block after the failed one depends on it, so none can be trusted
        untrusted = [b.index for b in self._chain[block.index:]]
        return {
            "valid": False,
            "failed_block_index": block.index,
            "failed_block_position": block.index + 1,
            "reason": reason,
            "stored_value": stored,
            "recomputed_value": expected,
            "untrusted_blocks": untrusted,
            "blocks_checked": block.index + 1,
        }

    def print_chain(self):
        for block in self._chain:
            print(block)

    @staticmethod
    def print_report(report):
        print("=" * 70)
        if report["valid"]:
            print(f"VALIDATION RESULT: VALID  ({report['blocks_checked']} blocks checked)")
        else:
            print("VALIDATION RESULT: INVALID - tampering detected")
            print(f"  First failing block : Block #{report['failed_block_index']} "
                  f"(block {report['failed_block_position']} in chain)")
            print(f"  Reason              : {report['reason']}")
            print(f"  Stored value        : {report['stored_value']}")
            print(f"  Recomputed value    : {report['recomputed_value']}")
            print(f"  Untrusted blocks    : {report['untrusted_blocks']}")
        print("=" * 70)

In [ ]:
# ======================= Test run =======================
bc = Blockchain()

# 9 more blocks -> 10 total including genesis, each with different transactions
sample_data = [
    [{"sender": "Alice", "receiver": "Bob", "amount": 10}],
    [{"sender": "Bob", "receiver": "Charlie", "amount": 5},
     {"sender": "Charlie", "receiver": "Dave", "amount": 2}],
    [{"sender": "Dave", "receiver": "Eve", "amount": 7}],
    [{"sender": "Eve", "receiver": "Frank", "amount": 3},
     {"sender": "Frank", "receiver": "Grace", "amount": 1},
     {"sender": "Grace", "receiver": "Alice", "amount": 4}],
    [{"sender": "Heidi", "receiver": "Ivan", "amount": 12}],
    [{"sender": "Ivan", "receiver": "Judy", "amount": 6},
     {"sender": "Judy", "receiver": "Mallory", "amount": 8}],
    [{"sender": "Mallory", "receiver": "Niaj", "amount": 9}],
    [{"sender": "Niaj", "receiver": "Olivia", "amount": 11},
     {"sender": "Olivia", "receiver": "Peggy", "amount": 2}],
    [{"sender": "Peggy", "receiver": "Trent", "amount": 15}],
]
for txs in sample_data:
    bc.add_block(txs)

print("\n########## BLOCKCHAIN BEFORE MODIFICATION ##########\n")
bc.print_chain()
Blockchain.print_report(bc.validate())


########## BLOCKCHAIN BEFORE MODIFICATION ##########

Block #0 (block 1 in chain)
  Timestamp     : 2026-09-20 13:07:21
  Transactions  :
      - Network -> Genesis: 0
  Merkle Root   : b512386df69dda67a807a09f58780abc0c9953deeb134c65666adac1a96992ed
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 2d18725ff2a8d7f212e8a86b3735cc146631a6ab58c9d99286199b784a3bc752

Block #1 (block 2 in chain)
  Timestamp     : 2026-09-20 13:07:21
  Transactions  :
      - Alice -> Bob: 10
  Merkle Root   : 634205f68815df3adbe60b160bba9809e72b3fdf6ca15e3bf164508616fd7d6b
  Previous Hash : 2d18725ff2a8d7f212e8a86b3735cc146631a6ab58c9d99286199b784a3bc752
  Hash          : 93fe5ff66534c73ef06bf2cc81b39366454a81698fb42c38577d465edddc4729

Block #2 (block 3 in chain)
  Timestamp     : 2026-09-20 13:07:21
  Transactions  :
      - Bob -> Charlie: 5
      - Charlie -> Dave: 2
  Merkle Root   : cb4d9bd31111f5326b3233f7e615fa149eba83dd791cb3a3c04887180a25cf6a
  

In [ ]:
# ---- Task 6: tamper with the 5th block (index 4) ----
# Change tx data only; do NOT recompute merkle_root or hash.
fifth = bc.chain[4]
print("########## TAMPERING WITH 5th BLOCK (index 4) ##########")
print(f"Original tx : {fifth.transactions}")
fifth.transactions[0]["amount"] = 1000
print(f"Modified tx : {fifth.transactions}\n")

print("########## BLOCKCHAIN AFTER MODIFICATION ##########\n")
bc.print_chain()
Blockchain.print_report(bc.validate())

########## TAMPERING WITH 5th BLOCK (index 4) ##########
Original tx : [{'sender': 'Eve', 'receiver': 'Frank', 'amount': 3}, {'sender': 'Frank', 'receiver': 'Grace', 'amount': 1}, {'sender': 'Grace', 'receiver': 'Alice', 'amount': 4}]
Modified tx : [{'sender': 'Eve', 'receiver': 'Frank', 'amount': 1000}, {'sender': 'Frank', 'receiver': 'Grace', 'amount': 1}, {'sender': 'Grace', 'receiver': 'Alice', 'amount': 4}]

########## BLOCKCHAIN AFTER MODIFICATION ##########

Block #0 (block 1 in chain)
  Timestamp     : 2026-09-20 13:07:21
  Transactions  :
      - Network -> Genesis: 0
  Merkle Root   : b512386df69dda67a807a09f58780abc0c9953deeb134c65666adac1a96992ed
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 2d18725ff2a8d7f212e8a86b3735cc146631a6ab58c9d99286199b784a3bc752

Block #1 (block 2 in chain)
  Timestamp     : 2026-09-20 13:07:21
  Transactions  :
      - Alice -> Bob: 10
  Merkle Root   : 634205f68815df3adbe60b160bba9809e72b3fd

In [ ]:
# ---- Extra evidence: attacker also recomputes block 5's Merkle Root + hash ----
# Shows why later blocks break: block 6's previous_hash no longer matches.
fifth.merkle_root = fifth.compute_merkle_root()
fifth.hash = fifth.compute_hash()
print("########## EXTRA: ATTACKER RECOMPUTES BLOCK 5 ROOT + HASH ##########")
Blockchain.print_report(bc.validate())

########## EXTRA: ATTACKER RECOMPUTES BLOCK 5 ROOT + HASH ##########
VALIDATION RESULT: INVALID - tampering detected
  First failing block : Block #5 (block 6 in chain)
  Reason              : Broken link - previous_hash does not match parent block's hash
  Stored value        : 128c4b5844f9c1bce7d215f8335856d9453a258a81fbc9da82414eeb8c34ff25
  Recomputed value    : 0af9714139221f86aa0b86d1fc92d8535ec7c98f865e54893d69b2a57aa889da
  Untrusted blocks    : [5, 6, 7, 8, 9]


In [ ]:
# ---- Extra: time a full rewrite of blocks 5-10 ----
# Evidence for Q3: without Proof of Work, hiding tampering costs almost nothing.
start = time.perf_counter()
for i in range(4, len(bc.chain)):
    blk = bc.chain[i]
    if i > 4:
        # re-link to the rewritten parent, otherwise the chain stays broken
        blk.previous_hash = bc.chain[i - 1].hash
    blk.merkle_root = blk.compute_merkle_root()
    blk.hash = blk.compute_hash()
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Attacker rewrote blocks 5-10 in {elapsed_ms:.3f} ms")
Blockchain.print_report(bc.validate())   # expected: VALID -> tampering is now hidden

Attacker rewrote blocks 5-10 in 4.795 ms
VALIDATION RESULT: VALID  (10 blocks checked)
